# Hands-on: Constraint-based modelling of metabolism

Computational Systems Biology (42589), Master's in Bioinformatics, ETSE, Universitat de
Valencia. **Session 10: metabolic modelling and flux balance analysis.**

Material mainly developed by Miguel Ponce de Leon (Barcelona Supercomputing Center) and
adapted by Arnau Montagud (I2SysBio, CSIC-UV).

## 1. Introduction

In the previous two sessions you worked with Boolean models, where each node was on or off
and time was discrete. Metabolism needs a different formalism: here we care about *rates*
(fluxes) through a few thousand reactions, we have no kinetic parameters for almost any of
them, and we still want quantitative predictions.

Constraint-based modelling gets around the missing parameters by refusing to compute a single
trajectory. Instead it describes the whole space of flux distributions that are compatible
with (i) mass balance at steady state and (ii) the capacity limits of each reaction, and then
picks one point in that space by optimising something, usually growth. That is flux balance
analysis (FBA, *analisis de balance de flujos*).

By the end of the session you should be able to:

1. Build a small stoichiometric model, write down its stoichiometric matrix (*matriz
   estequiometrica*) and explain what S * v = 0 and the flux bounds mean.
2. Run an FBA on a genome-scale model (*modelo a escala genomica*), change the growth medium
   and read the solution, knowing that the solution need not be unique.
3. Predict gene essentiality (*esencialidad genica*) in silico and judge those predictions
   against experimental data with a confusion matrix.
4. Use FBA for strain design: read a production envelope and couple a product to growth.

## 2. Material and methods

### 2.1 Where you are running this

You are almost certainly in **Binder**, which gives you a temporary machine in the cloud with
everything already installed.

- **Binder keeps nothing.** Download your notebook with `File > Download` now, and again at
  the end of the session. The copy you download is the one you hand in.
- **Ten minutes without activity closes your session.** Download before the break.

If you prefer to work on your own machine, `INSTALL.md` in the repository explains how.

### 2.2 The models

- A **toy model** with three metabolites and seven reactions, which we build from scratch so
  that nothing about the formalism stays hidden.
- **iJO1366**, the genome-scale reconstruction of *Escherichia coli* K-12 MG1655 [1]. It has
  2583 reactions, 1805 metabolites and 1367 genes, and it comes bundled with COBRApy, so no
  download is needed.

### 2.3 The software

We use **COBRApy** [2], the Python implementation of the COBRA methods. The linear programs
are solved with GLPK.

In [ ]:
import json
import warnings

import cobra
import matplotlib.pyplot as plt
import pandas as pd
from cobra import Metabolite, Model, Reaction
from cobra.flux_analysis import (
    flux_variability_analysis,
    production_envelope,
    single_gene_deletion,
)
from cobra.util.array import create_stoichiometric_matrix

# iJO1366 is an old reconstruction and loading it prints a long list of SBML notices.
warnings.filterwarnings("ignore")

import importlib.metadata as _md

for _pkg in ("cobra", "pandas", "matplotlib", "swiglpk"):
    try:
        print(f"{_pkg:12s} {_md.version(_pkg)}")
    except Exception:
        print(f"{_pkg:12s} version not reported")

Full COBRApy documentation: <https://cobrapy.readthedocs.io/en/latest/>

---

# Hands-on 1: from a toy model to a genome-scale model

## 3.1 Building a toy model from scratch

![toy model](img/toy_model.png)

**Figure 1.** Toy model with three metabolites (A, B and C), four internal reactions (v1 to
v4) and three exchange fluxes (b1 to b3). Left: the reaction scheme. Right: the system of
mass balances, written as the stoichiometric matrix N (which we will call S) multiplied by
the flux vector v.

A model in COBRApy is a container of `Metabolite` and `Reaction` objects. We start with the
container and two of the three metabolites.

In [ ]:
toy_model = Model("Toymodel")

A = Metabolite("A")
A.compartment = "cytosol"

B = Metabolite("B")
B.compartment = "cytosol"

### Exercise 1.1

Create metabolite `C`, in the same compartment as the other two.

In [ ]:
######################
# Create metabolite C
######################

## TODO
## Write your code below

In [ ]:
# Add the metabolites to the model
toy_model.add_metabolites([A, B, C])

# Print the reactions a given metabolite takes part in.
# We get an empty set because we have not created any reaction yet.
print(toy_model.metabolites.A.reactions)

Now the reactions. To add metabolites to a reaction we pass a dictionary whose keys are the
metabolites and whose values are the stoichiometric coefficients: negative for substrates,
positive for products.

Compare each line with Figure 1 as you read it.

In [ ]:
# b1 brings A into the system
b1 = Reaction("b1")
b1.add_metabolites({A: 1})

# b2 and b3 take B and C out of it
b2 = Reaction("b2")
b2.add_metabolites({B: -1})

b3 = Reaction("b3")
b3.add_metabolites({C: -1})

# the internal reactions
v1 = Reaction("v1")
v1.add_metabolites({A: -1, B: 1})

v2 = Reaction("v2")
v2.add_metabolites({A: -1, C: 1})

v3 = Reaction("v3")
v3.add_metabolites({A: 1, C: -1})

# v4 is exercise 1.2

### Exercise 1.2

Create reaction `v4` and add its metabolites. Read its stoichiometry off Figure 1.

In [ ]:
########################
# Create reaction v4
########################

## TODO
## Write your code below

In [ ]:
toy_model.add_reactions([b1, b2, b3, v1, v2, v3, v4])
print(toy_model)

Models are exchanged between tools in **SBML** (*Systems Biology Markup Language*), the
standard you saw in the first session of the course. Writing the model out is one line.

In [ ]:
from cobra.io import write_sbml_model

write_sbml_model(toy_model, "out/toymodel.sbml")

# Optional: open out/toymodel.sbml in the file browser on the left and look at it.

## 3.2 The stoichiometric matrix

Everything the model knows about the network topology is in S: one row per metabolite, one
column per reaction. The steady-state assumption is the single equation S * v = 0.

### Exercise 1.3

Print the stoichiometric matrix of `toy_model` with `create_stoichiometric_matrix`, as a
`pandas` DataFrame with metabolite IDs as the index and reaction IDs as the columns, and
check it against Figure 1.

Hints:

- `create_stoichiometric_matrix(toy_model)` returns a plain array.
- `[m.id for m in toy_model.metabolites]` and `[r.id for r in toy_model.reactions]` give you
  the labels.

Then answer: does your matrix match the figure exactly? If not, what is different, and does
that difference change the model?

In [ ]:
## TODO
## Write your code below

**Your answers (Exercise 1.3)**

1. 

## 3.3 Optimisation: the first FBA

Two more ingredients turn the topology into a model we can optimise:

- **bounds**, the capacity limits of each reaction. Here we let at most 1 unit of A enter.
- **an objective**, the reaction whose flux we maximise. Here it is `b2`, the output of B.

In [ ]:
# Limit the input
toy_model.reactions.b1.upper_bound = 1

# Set the objective
toy_model.objective = "b2"

solution = toy_model.optimize()

print("Objective value: %.2f" % solution.objective_value)
print("Status: %s" % solution.status)
print()
print(solution.fluxes)

solution.fluxes.to_csv("out/toymodel_fba.tsv", sep="\t")

### Exercise 1.4: is that solution unique?

FBA returns *one* optimal flux distribution, but there can be many that reach the same
objective value. **Flux variability analysis** (FVA) answers the question properly: for each
reaction it computes the lowest and highest flux compatible with staying optimal.

1. Run `flux_variability_analysis(toy_model)` and look at the range of each reaction.
2. Which reactions are fixed by the optimum, and which are free to vary? Explain what the
   model is free to do, in terms of Figure 1.
3. Look at `v2` and `v3` in particular. Their range is enormous. What are they doing, and is
   it biologically meaningful?
4. Run it again with `loopless=True` and compare. What changed, and why?

In [ ]:
## TODO
## Write your code below

**Your answers (Exercise 1.4)**

1. 
2. 
3. 
4. 

## 3.4 The genome-scale model

`load_model` fetches a model by name. `iJO1366` ships with COBRApy, so this works without an
internet connection. The same function can pull any model from the BiGG database
(<http://bigg.ucsd.edu/models/iJO1366>).

In [ ]:
model = cobra.io.load_model("iJO1366")
print(model)

### Exercise 1.5: inspecting the model

1. How many reactions, metabolites and genes does it have? (`len(model.reactions)` and so on.)
2. Get the reaction `ENO` (enolase). Print its `name`, `bounds`, `reaction` and
   `gene_reaction_rule`. What does the GPR rule tell you?
   Hint: `model.reactions.ENO`, or the safer `model.reactions.get_by_id("ENO")`.
3. Get the gene `b0720` and print the reactions it is associated with
   (`[r.id for r in gene.reactions]`).
4. What is the objective of this model? Hint:
   `[r.id for r in model.reactions if r.objective_coefficient != 0]`.
5. Print `model.medium`. These are the nutrients the cell is allowed to take up, and the
   maximum uptake rate of each, in mmol per gram dry weight per hour. Which carbon source is
   it, and is there oxygen?

Note: gene *names* are not stored in this model, only the b-numbers of the K-12 genome. Look
a b-number up at <https://ecocyc.org> if you want to know what it codes for.

In [ ]:
## TODO
## Write your code below

**Your answers (Exercise 1.5)**

1. 
2. 
3. 
4. 
5. 

## 3.5 Running an FBA on iJO1366

The objective is `BIOMASS_Ec_iJO1366_core_53p95M`, an artificial reaction that drains
precursors (amino acids, nucleotides, lipids, ATP) in the proportions needed to make one gram
of new cell. Maximising it means maximising the growth rate, in units of 1/h.

A note on signs that confuses everyone once: `model.medium` lists uptake as a **positive**
number, but the underlying exchange reactions are written as `metabolite -->`, so uptake is a
**negative** flux. A glucose uptake of 10 appears as `EX_glc__D_e = -10`.

In [ ]:
solution = model.optimize()

print("Objective value: %.4f  (1/h)" % solution.objective_value)
print("Status: %s" % solution.status)
print()
print(model.summary())

solution.fluxes.to_csv("out/iJO1366_fba.tsv", sep="\t")

### Exercise 1.6

Print the flux of these three reactions in the solution you just computed, and say what each
number means (units included, and watch the signs):

- glucose uptake: `EX_glc__D_e`
- oxygen uptake: `EX_o2_e`
- growth: `BIOMASS_Ec_iJO1366_core_53p95M`

Hint: `solution.fluxes["EX_glc__D_e"]`.

In [ ]:
## TODO
## Write your code below

**Your answers (Exercise 1.6)**

1. 

## 3.6 Changing the medium: growth without oxygen

The medium is just a set of bounds, so changing the environment is changing numbers. We take
the oxygen away and see what the cell does. Note the `with model:` block: every change inside
it is undone on the way out, so we cannot corrupt the model by accident.

### Exercise 1.7

1. Inside the `with` block, set the uptake of `EX_o2_e` to 0 and optimise. By what factor
   does the growth rate drop?
2. Print the exchange fluxes that are positive in the anaerobic solution. Those are the
   products the cell secretes. Which are they, and what is this fermentation called?
3. Why can the cell not simply run the same pathways more slowly and grow at the same rate?

Hints:

- Change the medium with a copy: `medium = model.medium`, then `medium["EX_o2_e"] = 0`, then
  `model.medium = medium`.
- `model.summary()` already separates uptake from secretion.

In [ ]:
with model:
    ## TODO
    ## Write your code below
    pass

**Your answers (Exercise 1.7)**

1. 
2. 
3. 

---

# Hands-on 2: knock-out studies and model validation

A model is only worth something if its predictions can be wrong in a way we can detect. Gene
essentiality is the standard test: delete a gene in silico, see whether the model can still
grow, and compare against an experiment where that gene was actually deleted.

## 3.7 A single knock-out

`gene.knock_out()` sets to zero every reaction that the GPR rule can no longer support.

In [ ]:
gene = model.genes.get_by_id("b0720")

print("Reactions associated with b0720:")
for r in gene.reactions:
    print(f"  {r.id:8s} {r.name}")
    print(f"           {r.reaction}")
    print(f"           GPR: {r.gene_reaction_rule}")

with model:
    gene.knock_out()
    ko_solution = model.optimize()

print()
# The solver can return a tiny negative number instead of a clean zero.
ko_growth = max(0.0, ko_solution.objective_value)
print("Growth of the b0720 knock-out: %.4f 1/h" % ko_growth)
print("Growth of the wild type:       %.4f 1/h" % solution.objective_value)

b0720 is *gltA*, citrate synthase, the first step of the TCA cycle. The model says the cell
cannot grow without it on glucose minimal medium. We will check that against experiment in a
moment, together with the other 1366 genes.

## 3.8 The whole genome

`single_gene_deletion` repeats the operation for a list of genes. All 1367 take a few seconds.

In [ ]:
knockout = single_gene_deletion(model, gene_list=[g.id for g in model.genes])

# The 'ids' column holds one-element sets; pull the gene ID out of each.
knockout["gene"] = [
    list(ids)[0] if isinstance(ids, (set, frozenset)) else ids for ids in knockout["ids"]
]

print(knockout[["gene", "growth", "status"]].head())
print()
print("deletions computed:", len(knockout))
print("infeasible (growth is NaN):", int(knockout["growth"].isna().sum()))

Now we turn growth rates into a yes or no answer. A deletion counts as **lethal** if it leaves
the cell with less than 1% of the wild-type growth rate. A deletion that makes the problem
infeasible (`NaN`) is lethal too, and must not be dropped silently.

In [ ]:
wild_type_growth = solution.objective_value
threshold = 0.01 * wild_type_growth

all_genes = {g.id for g in model.genes}
insilico_lethals = set(
    knockout.loc[knockout["growth"].isna() | (knockout["growth"] < threshold), "gene"]
)
insilico_non_lethals = all_genes - insilico_lethals

print("threshold: %.4f 1/h" % threshold)
print("in silico lethals:    ", len(insilico_lethals))
print("in silico non-lethals:", len(insilico_non_lethals))

## 3.9 Comparing against experiment

`data/m9_invivo_lethals.json` holds the genes found to be essential experimentally in
*E. coli* grown on M9 glucose minimal medium, the same condition the model is in.

One assumption is worth stating out loud, because it will matter later: any model gene that is
**not** on that list is treated as non-essential. The experiment did not necessarily test
every gene.

In [ ]:
with open("data/m9_invivo_lethals.json") as fh:
    invivo_lethals = set(json.load(fh))

# Keep only genes that exist in the model, so both sets describe the same universe.
invivo_lethals = invivo_lethals & all_genes
invivo_non_lethals = all_genes - invivo_lethals

print("in vivo lethals:    ", len(invivo_lethals))
print("in vivo non-lethals:", len(invivo_non_lethals))
print()
print("Is b0720 essential in vivo?", "b0720" in invivo_lethals)

With the two classifications in hand, the four cells of a confusion matrix are four set
intersections. Taking "essential" as the positive class:

In [ ]:
TP = insilico_lethals & invivo_lethals          # predicted essential, essential in vivo
TN = insilico_non_lethals & invivo_non_lethals  # predicted non-essential, non-essential in vivo
FP = insilico_lethals & invivo_non_lethals      # predicted essential, but not essential in vivo
FN = insilico_non_lethals & invivo_lethals      # predicted non-essential, but essential in vivo

P = TP | FN  # all genes essential in vivo
N = TN | FP  # all genes non-essential in vivo

for label, s in (("TP", TP), ("TN", TN), ("FP", FP), ("FN", FN), ("P", P), ("N", N)):
    print(f"{label}: {len(s)}")

### Evaluated exercise E1 (20% of the task)

Fill in this table with your own numbers from the cell above, and compute the four standard
metrics in the code cell below: **sensitivity**, **specificity**, **precision** and
**accuracy**. The definitions are at
<https://en.wikipedia.org/wiki/Sensitivity_and_specificity>; write the formula you used next
to each one.

| in vivo \ in silico   | in silico lethal | in silico non-lethal |
| --------------------- |:----------------:| --------------------:|
| **in vivo lethal**    |        ?         |          ?           |
| **in vivo non-lethal**|        ?         |          ?           |

Then answer, in two or three sentences: which of the four numbers would you quote to someone
who asked "how good is this model at predicting essential genes?", and why that one?

In [ ]:
# Sensitivity, recall or true positive rate (TPR). This one is done for you:
sensitivity = len(TP) / len(P)

## TODO
## Complete the three that are missing

# Specificity, selectivity or true negative rate (TNR)
specificity = ...

# Precision or positive predictive value (PPV)
precision = ...

# Accuracy (ACC)
accuracy = ...

for name, value in (
    ("sensitivity", sensitivity),
    ("specificity", specificity),
    ("precision", precision),
    ("accuracy", accuracy),
):
    print(f"{name:12s} {value}")

**Your answers (Exercise E1)**

1. 

### Evaluated exercise E2 (20% of the task)

*We go through the idea in class; you produce the results at home.*

Essentiality is not a property of a gene, it is a property of a gene **in an environment**. A
gene whose product makes an amino acid is essential on minimal medium and dispensable when
that amino acid is in the medium.

`data/rich_medium.json` is a richer medium: the same M9 salts plus amino acids, nucleosides
and vitamins. It was built from `data/argonne_media.json`, the medium used in the experiment
whose essential genes are in `data/argonne_invivo_lethals.json`.

Repeat the whole analysis of sections 3.8 and 3.9 in this medium, inside a `with model:`
block, against `data/argonne_invivo_lethals.json`, and report:

1. The growth rate, and the number of in silico lethals, compared with the minimal medium.
2. The new confusion matrix and the four metrics.
3. Which classification errors got better, and which got worse? Explain why, in terms of what
   a richer medium does to the network.
4. Take `b0720` again. Is it essential in silico in this medium? Is it on the experimental
   list for this medium? Explain the pattern across the two media.

The loop runs over 1367 genes twice, so give it a few seconds.

In [ ]:
with open("data/rich_medium.json") as fh:
    rich_medium = json.load(fh)

with model:
    model.medium = rich_medium
    ## TODO
    ## Write your code below
    pass

**Your answers (Exercise E2)**

1. 
2. 
3. 
4. 

### Evaluated exercise E3 (30% of the task)

A number like "accuracy = 0.9" tells you nothing about *why* the model is wrong. This exercise
is about the mistakes themselves, on the **minimal medium** results from E1.

Pick **two false negatives** (genes essential in vivo that the model says are dispensable) and
**two false positives** (the other way round), and for each of the four explain the specific
reason, with evidence from the model. Useful things to look at:

- `model.genes.get_by_id(g).reactions` and the `gene_reaction_rule` of each of those reactions.
  An `or` in a GPR means the model has an alternative gene for that step.
- Which metabolites the reaction produces, and whether the biomass reaction needs them
  (`model.reactions.BIOMASS_Ec_iJO1366_core_53p95M.metabolites`).
- The b-number at <https://ecocyc.org>, for what the gene actually does.

Then answer:

1. Your four genes, with the reason for each (two or three sentences each).
2. Of the two error types, which one is mostly the model's fault and which one is mostly the
   experiment's or the medium's? Argue it.
3. We assumed every gene absent from the experimental list is non-essential. Which cell of the
   confusion matrix does that assumption inflate, and what would you need in order to avoid it?

The cell below prints your own false positives and false negatives to choose from.

In [ ]:
print("False negatives (%d): in vivo essential, model says dispensable" % len(FN))
print(sorted(FN)[:20], "...")
print()
print("False positives (%d): model says essential, dispensable in vivo" % len(FP))
print(sorted(FP)[:20], "...")
print()

# Example of how to look one of them up
example = sorted(FN)[0]
g = model.genes.get_by_id(example)
print(f"{example} takes part in:")
for r in g.reactions:
    print(f"  {r.id:10s} GPR: {r.gene_reaction_rule}")

In [ ]:
## TODO
## Use this cell to investigate your four genes

**Your answers (Exercise E3)**

1. 
2. 
3. 

---

# Hands-on 3: metabolic engineering

So far we have used FBA to describe what the cell does. Now we use it to decide what to
change. The goal of strain design is a cell that makes a product **because** it grows, not
instead of growing, so that selection works for us rather than against us.

We work anaerobically, where *E. coli* ferments glucose into a mixture of products, and we try
to push it towards **succinate**, a precursor of polymers and solvents.

## 3.10 Production envelopes

A production envelope plots, for every achievable growth rate, the minimum and maximum flux
through the product. It is the honest picture of the trade-off, where a single FBA solution
would only show one point of it.

Each marker in the plots below is one simulation: we fix the growth rate at that value and
solve twice, once minimising and once maximising succinate. The line between markers is
interpolation, not data.

The right-hand panel multiplies the two axes of the left one together. That product,
succinate flux times growth rate, is what a fermenter actually accumulates per hour: a fast
culture making a little and a slow culture making a lot can come out the same. Its peak,
marked with a star, is the growth rate you would aim for. It is usually called the
biomass-product coupled yield.

In [ ]:
BIOMASS = "BIOMASS_Ec_iJO1366_core_53p95M"

anaerobic_medium = model.medium
anaerobic_medium["EX_o2_e"] = 0.0

POINTS = 15  # how many growth rates we actually simulate


def succinate_envelope(m):
    return production_envelope(m, [BIOMASS], objective="EX_succ_e", points=POINTS)


def two_panels(title):
    fig, (ax_env, ax_prod) = plt.subplots(1, 2, figsize=(12, 4.5))
    ax_env.set(xlabel="growth rate (1/h)", ylabel="succinate secretion (mmol/gDW/h)",
               title="Production envelope")
    ax_prod.set(xlabel="growth rate (1/h)", ylabel="succinate flux x growth rate",
                title="Product of the two")
    fig.suptitle(title)
    return fig, (ax_env, ax_prod)


def draw(env, axes, label, colour):
    # Envelope on the left, succinate flux times growth rate on the right.
    # Each marker is one simulated point. The star marks where the product peaks.
    ax_env, ax_prod = axes
    growth = env[BIOMASS]

    for column, marker, alpha in (("flux_maximum", "-o", 1.0), ("flux_minimum", "--s", 0.55)):
        ax_env.plot(growth, env[column], marker, ms=4, color=colour, alpha=alpha,
                    label=f"{label}, {column.split('_')[1]}")

        product = env[column] * growth
        ax_prod.plot(growth, product, marker, ms=4, color=colour, alpha=alpha,
                     label=f"{label}, {column.split('_')[1]}")

        peak = product.idxmax()
        if product[peak] > 1e-6:
            ax_prod.plot(growth[peak], product[peak], "*", ms=16, color=colour, alpha=alpha)
            offset = (6, 6) if column == "flux_maximum" else (6, -14)
            ax_prod.annotate(f"{product[peak]:.2f} at {growth[peak]:.3f}",
                             (growth[peak], product[peak]), textcoords="offset points",
                             xytext=offset, fontsize=8, color=colour)


with model:
    model.medium = anaerobic_medium
    wt_env = succinate_envelope(model)

fig, axes = two_panels("Wild type, anaerobic")
draw(wt_env, axes, "wild type", "tab:blue")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(wt_env[[BIOMASS, "flux_minimum", "flux_maximum"]].round(3).tail(3))

Read the right-hand edge of that plot. At the maximum growth rate the minimum and the maximum
succinate flux are the same small number: a cell that grows as fast as it can makes almost no
succinate. The capacity is there (the top left of the envelope), but nothing forces the cell
to use it.

## 3.11 Coupling the product to growth

The classic move is to delete the competing fermentation routes, so that the only way left to
balance redox is through succinate. We do it one reaction at a time and watch the envelope
close.

### Exercise 3.1

In the cell below, knock out the three competing routes inside a `with` block and redraw the
envelope on top of the wild-type one:

- `ALCD2x`, alcohol dehydrogenase, the route to ethanol
- `LDH_D`, lactate dehydrogenase, the route to lactate
- `PFL`, pyruvate formate lyase, the route to formate

Then answer:

1. What is the maximum growth rate now, compared with the wild type?
2. At that maximum growth rate, what is the **minimum** succinate flux? That number is the
   point of the whole exercise. Explain why.
3. Divide it by the glucose uptake (10 mmol/gDW/h) to get the yield in mol per mol. How far is
   it from the theoretical maximum at zero growth?
4. Look at the right-hand panel, the product of flux and growth rate. For the wild type, the
   curve built from the minimum is flat on zero at every growth rate. For the knock-out it has
   a peak. Say what that difference means for someone running the fermenter.
5. Would you expect the real strain to behave exactly like this? Name one reason it might not.

In [ ]:
with model:
    model.medium = anaerobic_medium
    ## TODO
    ## Knock out the three reactions, then draw the envelope
    pass

**Your answers (Exercise 3.1)**

1. 
2. 
3. 
4. 
5. 

### Exercise 3.2: why reactions and not genes?

In hands-on 2 we knocked out *genes*. Here we knocked out *reactions*. That was not an
accident.

Run the cell below, which deletes the genes usually named as the targets of this design
(*adhE*, *ldhA* and *pflB*) and recomputes the growth and the succinate flux. Then explain
what you see, using the GPR rules it prints.

This is the same phenomenon you were asked to find in E3. Say which of the two error types it
produces.

In [ ]:
with model:
    model.medium = anaerobic_medium
    for rid in ("ALCD2x", "LDH_D", "PFL"):
        print(f"{rid:8s} GPR: {model.reactions.get_by_id(rid).gene_reaction_rule}")
    print()

    for gene_id in ("b1241", "b1380", "b0903"):  # adhE, ldhA, pflB
        model.genes.get_by_id(gene_id).knock_out()

    gene_ko = model.optimize()
    print("growth after deleting adhE, ldhA and pflB: %.4f 1/h" % gene_ko.objective_value)
    print("succinate secretion: %.2f mmol/gDW/h" % gene_ko.fluxes["EX_succ_e"])

**Your answers (Exercise 3.2)**

1. 

### Evaluated exercise E4 (30% of the task)

Design a producer strain of your own, and support it with simulations. You may stay in
iJO1366 or use another model (`cobra.io.load_model` takes any BiGG identifier, for example
`iML1515` for a newer *E. coli*, `iMM904` for yeast; see <http://bigg.ucsd.edu/models>).

Choose a product and work through:

1. **The product and the host.** Which metabolite, which organism, and why that pairing. If
   the product is not already in the model, say which reactions and which enzymes, from which
   organism, would have to be added, and add them with `model.add_reactions`.
2. **The baseline.** The production envelope of the unmodified strain in the growth condition
   you choose. Is the product coupled to growth at all?
3. **The intervention.** At least one knock-out, addition or medium change, each one
   justified in terms of the metabolism and not by trial and error. Redraw the envelope.
4. **The outcome.** Growth rate, product flux at maximum growth and yield per unit of
   substrate, before and after. Say honestly whether your design improved anything.
5. **The limits.** Two things this analysis cannot tell you about whether the strain would
   work in a fermenter.

A design that fails and is correctly diagnosed is worth more than one that appears to succeed
without evidence. Keep it to the size of the exercises above, not a research project.

In [ ]:
## Your design goes here. Add as many cells as you need.

**Your answers (Exercise E4)**

1. 
2. 
3. 
4. 
5. 

---

## What to hand in

The notebook, executed, with:

- the code cells of exercises 1.1 to 1.7, 3.1 and 3.2 completed, with their output visible,
- the four evaluated exercises E1 to E4 answered, including the code they need,
- your written answers in the "Your answers" cells,
- the figures the simulations produce.

Download it with `File > Download` and upload it to Aula Virtual as
`Tarea7_TuApellido.ipynb`. The deadline and the marking criteria are in the task description
there.

## Before you close this tab

`File > Download` your notebook. Binder deletes everything when your session ends, and the
notebook you download is what you hand in.

## References

1. Orth JD, Conrad TM, Na J, Lerman JA, Nam H, Feist AM, Palsson BO. A comprehensive
   genome-scale reconstruction of *Escherichia coli* metabolism, 2011. *Molecular Systems
   Biology* 2011; 7: 535. <https://doi.org/10.1038/msb.2011.65>
2. Ebrahim A, Lerman JA, Palsson BO, Hyduke DR. COBRApy: COnstraints-Based Reconstruction and
   Analysis for Python. *BMC Systems Biology* 2013; 7: 74.
   <https://doi.org/10.1186/1752-0509-7-74>
3. Orth JD, Thiele I, Palsson BO. What is flux balance analysis? *Nature Biotechnology* 2010;
   28: 245-248. <https://doi.org/10.1038/nbt.1614>
4. Burgard AP, Pharkya P, Maranas CD. OptKnock: a bilevel programming framework for
   identifying gene knockout strategies for microbial strain optimization. *Biotechnology and
   Bioengineering* 2003; 84: 647-657. <https://doi.org/10.1002/bit.10803>
5. Jantama K, Haupt MJ, Svoronos SA, Zhang X, Moore JC, Shanmugam KT, Ingram LO. Combining
   metabolic engineering and metabolic evolution to develop nonrecombinant strains of
   *Escherichia coli* C that produce succinate and malate. *Biotechnology and Bioengineering*
   2008; 99: 1140-1153. <https://doi.org/10.1002/bit.21694>